# preflight Demo on Public Datasets

This notebook shows policy-based readiness checks on public datasets with richer visual output.

In [1]:
from sklearn.datasets import load_breast_cancer, load_iris, load_wine
from sklearn.model_selection import train_test_split

import pandas as pd
from IPython.display import HTML, Markdown, display

import preflight

In [2]:
def as_frame(dataset_loader):
    ds = dataset_loader(as_frame=True)
    df = ds.frame.copy()
    target_name = ds.target.name if ds.target is not None and ds.target.name else "target"
    if target_name not in df.columns:
        df[target_name] = ds.target
    return df, target_name


def summarize_report(name, report, split_report):
    return {
        "dataset": name,
        "gate": report.gate.status,
        "score": round(report.score, 1),
        "findings": len(report.findings),
        "split_gate": split_report.gate.status,
        "split_score": round(split_report.score, 1),
        "split_findings": len(split_report.findings),
    }


def run_demo(name, loader):
    df, target = as_frame(loader)
    report_explore = preflight.run(df, target=target, profile="exploratory")
    report_balanced = preflight.run(df, target=target, profile="ci-balanced")

    train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df[target])
    split_report = preflight.run_split(train_df, test_df, profile="ci-balanced")

    display(Markdown(f"## {name}"))
    display(
        pd.DataFrame(
            [
                {
                    "rows": len(df),
                    "columns": len(df.columns),
                    "target": target,
                    "exploratory gate": report_explore.gate.status,
                    "ci-balanced gate": report_balanced.gate.status,
                    "split gate": split_report.gate.status,
                }
            ]
        )
    )

    display(Markdown("### Rich HTML report (ci-balanced)"))
    display(HTML(report_balanced.to_html()))

    display(Markdown("### Markdown preview"))
    md_preview = "\n".join(report_balanced.to_markdown().splitlines()[:18])
    display(Markdown(f"```markdown\n{md_preview}\n```"))

    return summarize_report(name, report_balanced, split_report), report_balanced

In [3]:
summary_rows = []
reports = {}

for name, loader in [
    ("Breast Cancer (sklearn/UCI)", load_breast_cancer),
    ("Wine (sklearn/UCI)", load_wine),
    ("Iris (sklearn/UCI)", load_iris),
]:
    row, report = run_demo(name, loader)
    summary_rows.append(row)
    reports[name] = report

display(Markdown("## Combined Summary"))
display(pd.DataFrame(summary_rows))

## Breast Cancer (sklearn/UCI)

,rows,columns,target,exploratory gate,ci-balanced gate,split gate
0,569,31,target,PASS,FAIL,PASS


### Rich HTML report (ci-balanced)

Severity,Check ID,Domain,Title,Confidence,Suppressed
info,balance.balanced,data_quality,Class distribution is acceptably balanced (ratio 1.7:1).,0.90,no
info,completeness.missingness,data_quality,Overall missingness is 0.0% (0 cells missing across dataset).,0.95,no
error,correlations.feature_pairs,stat_anomaly,15 highly correlated feature pair(s) detected.,0.85,no
info,dataset.fingerprint,advisory,Dataset fingerprint computed.,1.00,no
info,distributions.health,stat_anomaly,No major distributional health issues detected.,0.85,no
info,duplicates.exact,data_quality,No exact duplicate rows detected.,0.95,no
info,leakage.high_correlation,target_risk,No strong feature-target leakage signal detected.,0.80,no
info,types.sanity,schema_contract,No object/string columns found.,1.00,no


### Markdown preview

...

Severity,Check ID,Domain,Title,Confidence,Suppressed
info,balance.balanced,data_quality,Class distribution is acceptably balanced (ratio 1.0:1).,0.90,no
info,completeness.missingness,data_quality,Overall missingness is 0.0% (0 cells missing across dataset).,0.95,no
error,correlations.feature_pairs,stat_anomaly,1 highly correlated feature pair(s) detected.,0.85,no
info,dataset.fingerprint,advisory,Dataset fingerprint computed.,1.00,no
info,distributions.health,stat_anomaly,No major distributional health issues detected.,0.85,no
warn,duplicates.exact,data_quality,1 exact duplicate row(s) detected (0.7%).,0.95,no
critical,leakage.high_correlation,target_risk,Potential target leakage detected via feature-target correlation.,0.90,no
info,types.sanity,schema_contract,No object/string columns found.,1.00,no


### Markdown preview

```markdown
## Preflight Run Report

- **Gate:** `FAIL`
- **Heuristic score:** `80.0/100`
- **Profile:** `ci-balanced`
- **Rows analyzed:** `150` / `150`
- **Columns:** `5`
- **Target:** `target`

### Findings

| Severity | Check | Domain | Title |
|---|---|---|---|
| `info` | `balance.balanced` | `data_quality` | Class distribution is acceptably balanced (ratio 1.0:1). |
| `info` | `completeness.missingness` | `data_quality` | Overall missingness is 0.0% (0 cells missing across dataset). |
| `error` | `correlations.feature_pairs` | `stat_anomaly` | 1 highly correlated feature pair(s) detected. |
| `info` | `dataset.fingerprint` | `advisory` | Dataset fingerprint computed. |
| `info` | `distributions.health` | `stat_anomaly` | No major distributional health issues detected. |
```

## Combined Summary

,dataset,gate,score,findings,split_gate,split_score,split_findings
0,Breast Cancer (sklearn/UCI),FAIL,94.0,8,PASS,98.0,2
1,Wine (sklearn/UCI),PASS,100.0,8,FAIL,94.0,2
2,Iris (sklearn/UCI),FAIL,80.0,8,FAIL,94.0,2


In [4]:
payload = reports["Breast Cancer (sklearn/UCI)"].to_dict()
payload["schema_version"], payload["gate"], len(payload["findings"])

('2.0.0',
 {'status': 'FAIL',
  'reasons': ['1 error finding(s) exceeded policy fail threshold']},
 8)